# 20. 🧰 Прикладное: поиск, эмбеддинги и RAG

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/justxor/math-for-ai-2026/blob/main/notebooks/20_retrieval_rag.ipynb)

Код из раздела [modules/20_retrieval_rag.md](https://github.com/justxor/math-for-ai-2026/blob/main/modules/20_retrieval_rag.md#m20). Запускайте ячейки сверху вниз: последующие используют переменные предыдущих. Теория, формулы и картинки — в тексте курса по ссылке.

In [ ]:
# Colab: всё нужное уже установлено. Локально: pip install -r requirements.txt
import numpy as np
import math
np.set_printoptions(precision=4, suppress=True)

### 🏋️ Практика модуля

**20.1.** Реализуйте BM25 и найдите лучший документ для запроса.

*▶️ Решение — сначала попробуйте сами.*

In [ ]:
import numpy as np, math
from collections import Counter
docs = ["градиентный спуск обновляет веса модели",
        "adam адаптивный оптимизатор градиентный",
        "кошки любят спать на солнце",
        "спуск с горы на лыжах"]
toks = [d.split() for d in docs]; N = len(docs); avgdl = np.mean([len(t) for t in toks])
df = Counter(w for t in toks for w in set(t))
def bm25(query, k1=1.2, b=0.75):
    scores = []
    for t in toks:
        tf = Counter(t); s = 0.0
        for w in query.split():
            if w in tf:
                idf = math.log((N - df[w] + 0.5) / (df[w] + 0.5) + 1)
                s += idf * tf[w] * (k1 + 1) / (tf[w] + k1 * (1 - b + b * len(t) / avgdl))
        scores.append(round(float(s), 3))
    return scores
print(bm25("градиентный спуск"))   # первый документ — максимальный скор

### 🏋️ Практика модуля

**20.2.** Сравните точный поиск и IVF (через k-means) по recall@10 и числу вычисленных расстояний.

*▶️ Решение — сначала попробуйте сами.*

In [ ]:
rng = np.random.default_rng(0)
X = rng.normal(size=(20_000, 64)).astype(np.float32); X /= np.linalg.norm(X, axis=1, keepdims=True)
q = X[:100] + 0.1 * rng.normal(size=(100, 64)).astype(np.float32)
exact = np.argsort(-(q @ X.T), axis=1)[:, :10]
C = X[rng.choice(len(X), 100, replace=False)]                  # 100 кластеров (упрощённо: 5 итераций k-means)
for _ in range(5):
    lab = np.argmax(X @ C.T, 1); C = np.array([X[lab == j].mean(0) for j in range(100)])
lab = np.argmax(X @ C.T, 1)
for nprobe in [1, 5, 20]:
    rec, cost = 0, 0
    for i in range(100):
        cl = np.argsort(-(q[i] @ C.T))[:nprobe]; cand = np.where(np.isin(lab, cl))[0]; cost += len(cand)
        top = cand[np.argsort(-(X[cand] @ q[i]))[:10]]; rec += len(set(top) & set(exact[i])) / 10
    print(f"nprobe={nprobe:2d}: recall@10={rec / 100:.2f}, расстояний на запрос ≈ {cost // 100}")

### 🏋️ Практика модуля

**20.3.** Покажите на числах, что для нормированных векторов $\lVert\mathbf{a} - \mathbf{b}\rVert^2 = 2 - 2\cos(\mathbf{a}, \mathbf{b})$.

*▶️ Решение — сначала попробуйте сами.*

In [ ]:
a, b = rng.normal(size=8), rng.normal(size=8); a /= np.linalg.norm(a); b /= np.linalg.norm(b)
print(np.sum((a - b) ** 2), 2 - 2 * a @ b)   # совпадают

## 🔬 Лаборатория: иллюстрации модуля

Это код картинок из курса. Меняйте параметры (learning rate, температуру, число точек, ядро…) и перезапускайте ячейку — так интуиция появляется быстрее всего.

In [ ]:
# Помощники для иллюстраций (те же, что в scripts/make_figures.py)
import matplotlib.pyplot as plt
from scipy import stats
C = ["#2563eb", "#dc2626", "#059669", "#d97706", "#7c3aed", "#0891b2"]
plt.rcParams.update({"figure.dpi": 100, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": 0.25})

def save(fig, name):          # в ноутбуке просто показываем картинку
    fig.tight_layout(); plt.show()

def arrow(ax, v, color, label=None, origin=(0, 0), **kw):
    ax.annotate("", xy=(origin[0] + v[0], origin[1] + v[1]), xytext=origin,
                arrowprops=dict(arrowstyle="-|>", color=color, lw=2.2, **kw))
    if label:
        ax.text(origin[0] + v[0] * 1.08, origin[1] + v[1] * 1.08, label, color=color, fontsize=13, fontweight="bold")

def rosen_like(x, y):
    return 0.5 * x**2 + 4 * y**2

In [ ]:
def bm25():
    fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
    tf = np.arange(0, 21)
    axes[0].plot(tf, tf, "--", color="gray", label="сырая частота TF")
    for k1, c in zip([0.5, 1.2, 2.0], C):
        axes[0].plot(tf, tf * (k1 + 1) / (tf + k1), color=c, lw=2, label=f"BM25, k₁ = {k1}")
    axes[0].set_ylim(0, 8); axes[0].set_xlabel("сколько раз слово встретилось в документе")
    axes[0].legend(fontsize=9); axes[0].set_title("Насыщение: 20-е повторение почти ничего не добавляет")
    N = 1_000_000; df = np.logspace(0, 6, 200)
    axes[1].semilogx(df, np.log((N - df + 0.5) / (df + 0.5) + 1), color=C[1], lw=2.5)
    for w, d in [("«трансформер»", 300), ("«модель»", 40_000), ("«и»", 900_000)]:
        v = np.log((N - d + 0.5) / (d + 0.5) + 1); axes[1].plot(d, v, "o", color="k"); axes[1].text(d * 1.3, v + 0.3, w)
    axes[1].set_xlabel("в скольких документах из 10⁶ встречается слово (df)"); axes[1].set_ylabel("IDF")
    axes[1].set_title("IDF: редкие слова весят больше")
    save(fig, "bm25")

bm25()